# Crossmatch: MuSCAT-db targets vs. TESS TOIs

Which targets already observed by MuSCAT are TESS Objects of Interest
(TOIs), and what is their vetting disposition? This notebook does a spatial
crossmatch between the observed targets in the live `muscat.db` and the TOI
catalog (`data/TOIs.csv`), using `astropy.coordinates` for robust
coordinate handling.

**Environment:** run with the `prose` conda kernel. Data comes straight
from `muscat.db` (via `sqlite3`) and the bundled TOI CSV, so no external
network is needed.

In [ ]:
import os
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord, Angle
import astropy.units as u

SEP_ARCSEC = 5.0   # match radius


def find_db():
    """Read-only path to a populated muscat.db (env, then repo root)."""
    cands = []
    if os.environ.get("MUSCAT_DB_PATH"):
        cands.append(Path(os.environ["MUSCAT_DB_PATH"]))
    cands += [Path("../muscat.db"), Path("muscat.db")]
    for p in cands:
        if not p.exists():
            continue
        try:
            con = sqlite3.connect(f"file:{p}?mode=ro", uri=True)
            n = con.execute("SELECT count(*) FROM targets").fetchone()[0]
            con.close()
        except sqlite3.Error:
            continue
        if n > 0:
            return p.resolve()
    return None


DB = find_db()
TOI_CSV = next((p for p in [Path("../data/TOIs.csv"), Path("data/TOIs.csv")] if p.exists()), None)
print("muscat.db:", DB)
print("TOI csv  :", TOI_CSV)

## 1. MuSCAT targets from the live database

The `targets` table stores RA/Dec in sexagesimal strings (RA in hours, Dec
in degrees). We parse them to decimal degrees with `astropy.Angle` and drop
any rows that fail to parse.

In [ ]:
def to_degrees(ra_strings, dec_strings):
    """Sexagesimal RA (hourangle) / Dec (deg) strings -> decimal degrees."""
    ra_deg, dec_deg = [], []
    for ra_s, dec_s in zip(ra_strings, dec_strings):
        try:
            ra_deg.append(Angle(str(ra_s), unit=u.hourangle).deg)
            dec_deg.append(Angle(str(dec_s), unit=u.deg).deg)
        except Exception:
            ra_deg.append(np.nan)
            dec_deg.append(np.nan)
    return np.array(ra_deg), np.array(dec_deg)


con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
muscat = pd.read_sql_query(
    "SELECT object, ra, declination, n_frames, instruments FROM targets "
    "WHERE is_identified = 1 AND ra != '' AND declination != ''",
    con,
)
con.close()

muscat["ra_deg"], muscat["dec_deg"] = to_degrees(muscat["ra"], muscat["declination"])
muscat = muscat.dropna(subset=["ra_deg", "dec_deg"]).reset_index(drop=True)
print(f"{len(muscat)} identified MuSCAT targets with valid coordinates")
muscat.head()

## 2. The TESS TOI catalog

`data/TOIs.csv` is an export of the TESS Objects of Interest table. We keep
the identifier, coordinates, vetting disposition, and a few planet
parameters.

In [ ]:
toi = pd.read_csv(TOI_CSV)
keep = ["toi", "tic id", "tfopwg disposition", "ra_deg", "dec_deg",
        "period (days)", "depth (ppm)", "planet radius (r_earth)"]
toi = toi[[c for c in keep if c in toi.columns]].copy()
toi = toi.dropna(subset=["ra_deg", "dec_deg"]).reset_index(drop=True)
print(f"{len(toi)} TOIs with coordinates")
print("dispositions:", dict(toi["tfopwg disposition"].value_counts()))

## 3. Spatial crossmatch

`search_around_sky` returns every pair of (TOI, MuSCAT) sources within the
match radius. A few arcseconds is enough: MuSCAT pointings are centered on
the catalog position, so a genuine match is essentially coincident.

In [ ]:
c_muscat = SkyCoord(muscat["ra_deg"].values * u.deg, muscat["dec_deg"].values * u.deg)
c_toi = SkyCoord(toi["ra_deg"].values * u.deg, toi["dec_deg"].values * u.deg)

i_toi, i_muscat, d2d, _ = c_muscat.search_around_sky(c_toi, SEP_ARCSEC * u.arcsec)

print(f"match radius           : {SEP_ARCSEC} arcsec")
print(f"matched pairs          : {len(i_muscat)}")
print(f"distinct MuSCAT targets: {len(set(i_muscat))}")
print(f"distinct TOIs          : {len(set(i_toi))}")

## 4. Which MuSCAT targets are TOIs

Assemble the matched pairs into a table, sorted by separation.

In [ ]:
matches = pd.DataFrame({
    "muscat_object": muscat["object"].values[i_muscat],
    "instruments": muscat["instruments"].values[i_muscat],
    "n_frames": muscat["n_frames"].values[i_muscat],
    "toi": toi["toi"].values[i_toi],
    "disposition": toi["tfopwg disposition"].values[i_toi],
    "period_d": toi["period (days)"].values[i_toi],
    "depth_ppm": toi["depth (ppm)"].values[i_toi],
    "sep_arcsec": d2d.arcsec,
})
matches = matches.sort_values("sep_arcsec").reset_index(drop=True)

print(f"{len(matches)} matched pairs "
      f"({matches['muscat_object'].nunique()} distinct MuSCAT targets)\n")
with pd.option_context("display.max_rows", 20, "display.width", 120):
    print(matches.head(20).to_string(index=False))

## 5. Visualize on the sky

All MuSCAT targets and all TOIs, with the matched MuSCAT targets circled.
RA increases to the left, following the usual sky convention.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(toi["ra_deg"], toi["dec_deg"], s=4, color="0.8", label="TOIs")
ax.scatter(muscat["ra_deg"], muscat["dec_deg"], s=12, color="C0", alpha=0.7,
           label="MuSCAT targets")
matched_idx = sorted(set(i_muscat))
ax.scatter(muscat["ra_deg"].values[matched_idx], muscat["dec_deg"].values[matched_idx],
           s=60, facecolors="none", edgecolors="C3", linewidths=1.4, label="matched")
ax.set_xlabel("RA (deg)")
ax.set_ylabel("Dec (deg)")
ax.invert_xaxis()
_ = ax.set_title("MuSCAT targets that are TESS TOIs")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

## 6. Export

Write the matched pairs next to the input catalogs for follow-up.

In [ ]:
out = (TOI_CSV.parent / "crossmatch_muscat_toi.csv")
matches.to_csv(out, index=False)
print(f"wrote {len(matches)} rows to {out}")

## Summary

- MuSCAT target coordinates come from the live `muscat.db`; the TOI
  positions from the bundled catalog.
- `astropy` `search_around_sky` gives a robust spatial match in a few lines.
- The result flags which observed fields are TOIs and their disposition
  (PC = planet candidate, CP/KP = confirmed/known planet, FP = false
  positive), a useful filter when prioritizing analysis.
- Swap `TOIs.csv` for any catalog with RA/Dec (e.g. confirmed planets in
  `data/nexsci_pscomppars.csv`) to reuse the same workflow.